In [1]:
import numpy as np
from dataclasses import dataclass
from typing import Tuple, Dict, List

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import KFold
from scipy.stats import norm

import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# WEEK 9 — FUNCTION 3 (Scaling/Emergence-aware v3)
# Key upgrades vs Week 8:
#  - Small ENSEMBLE of MC-dropout surrogates (robustness vs emergent quirks)
#  - Global Sobol candidates + local trust-region candidates (cost-effective scaling)
#  - Risk-aware acquisition: EI + (mu + beta*sigma) hybrid
#  - Deterministic 6-decimal output, best-by-score, non-duplicate after rounding
# ============================================================

# ---------------------------------------------------------
# 1. Data: 3D inputs and 1D outputs (maximisation)
# ---------------------------------------------------------
X_raw = np.array([
    [0.17152521, 0.34391687, 0.2487372],
    [0.24211446, 0.64407427, 0.27243281],
    [0.53490572, 0.39850092, 0.17338873],
    [0.49258141, 0.61159319, 0.34017639],
    [0.13462167, 0.21991724, 0.45820622],
    [0.34552327, 0.94135983, 0.26936348],
    [0.15183663, 0.43999062, 0.99088187],
    [0.64550284, 0.39714294, 0.91977134],
    [0.74691195, 0.28419631, 0.22629985],
    [0.17047699, 0.6970324 , 0.14916943],
    [0.22054934, 0.29782524, 0.34355534],
    [0.66601366, 0.67198515, 0.2462953 ],
    [0.04680895, 0.23136024, 0.77061759],
    [0.60009728, 0.72513573, 0.06608864],
    [0.96599485, 0.86111969, 0.56682913],
    [1.065994  , 1.041359  , 1.090881  ],  # historical out-of-bounds
    [0.403482  , 0.38217   , 0.489363  ],
    [3.98350e-01, 1.00000e-06, 5.43642e-01],
    [0.962851  , 0.987386  , 0.040875  ],
    [0.504564  , 0.348726  , 0.601264  ],
    [0.265159  , 0.286931  , 0.413777  ],
    [0.403756  , 0.381706  , 0.489738  ],
    [0.359927  , 0.175969  , 0.720956  ]
], dtype=float)

y_raw = np.array([
    -0.1121222,  -0.08796286, -0.11141465, -0.03483531, -0.04800758,
    -0.11062091, -0.39892551, -0.11386851, -0.13146061, -0.09418956,
    -0.04694741, -0.10596504, -0.11804826, -0.03637783, -0.05675837,
    -0.769427956661122, -0.03310307977430594, -0.09333459499358941,
    -0.07627377706316849, -0.05678719487656195, -0.03492633073917894,
    -0.009136026447950633, -0.14476549871155003
], dtype=float)

DEVICE = torch.device("cpu")  # set "cuda" if available


# ---------------------------------------------------------
# Utility: bounds + rounding for submission
# ---------------------------------------------------------
def clip_to_bounds(X: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    return np.clip(X, lower, upper)

def round6(x: np.ndarray) -> np.ndarray:
    return np.round(x.astype(float), 6)

def as_fixed6_list(x: np.ndarray) -> str:
    return f"[{x[0]:.6f}, {x[1]:.6f}, {x[2]:.6f}]"


# ---------------------------------------------------------
# 2. PyTorch MLP model
# ---------------------------------------------------------
class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes=(64, 64), dropout=0.15):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def _train_mlp(
    Xs: np.ndarray,
    ys: np.ndarray,
    hidden: Tuple[int, ...],
    dropout: float,
    lr: float,
    weight_decay: float,
    n_epochs: int,
    seed: int,
    tol: float = 1e-6,
    patience: int = 80,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = MLPRegressorTorch(
        input_dim=Xs.shape[1],
        hidden_sizes=hidden,
        dropout=dropout
    ).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)
    y_tensor = torch.from_numpy(ys.astype(np.float32)).view(-1, 1).to(DEVICE)

    best_loss = float("inf")
    bad = 0

    for _ in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        preds = model(X_tensor)
        loss = criterion(preds, y_tensor)
        loss.backward()
        optimizer.step()

        l = float(loss.item())
        if best_loss - l > tol:
            best_loss = l
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    return model


# ---------------------------------------------------------
# 3. MC Dropout surrogate (robust y scaling)
# ---------------------------------------------------------
@dataclass
class MCDropoutSurrogate:
    hidden_layer_sizes: Tuple[int, ...] = (64, 64)
    dropout: float = 0.15
    n_epochs: int = 1200
    lr: float = 1e-3
    weight_decay: float = 1e-6
    random_state: int = 0
    n_mc_samples: int = 128
    robust_y: bool = True

    def __post_init__(self):
        self.model = None
        self.x_scaler = StandardScaler()
        self.y_scaler = RobustScaler() if self.robust_y else StandardScaler()

    def fit(self, X: np.ndarray, y: np.ndarray, lower: np.ndarray, upper: np.ndarray):
        Xc = clip_to_bounds(X, lower, upper)
        Xs = self.x_scaler.fit_transform(Xc)
        ys = self.y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

        self.model = _train_mlp(
            Xs, ys,
            hidden=self.hidden_layer_sizes,
            dropout=self.dropout,
            lr=self.lr,
            weight_decay=self.weight_decay,
            n_epochs=self.n_epochs,
            seed=self.random_state,
        )

    def predict(self, X: np.ndarray, lower: np.ndarray, upper: np.ndarray, return_std: bool = False):
        Xc = clip_to_bounds(X, lower, upper)
        Xs = self.x_scaler.transform(Xc)
        X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)

        # Keep dropout ON at inference
        self.model.train()

        preds_scaled_mc = []
        with torch.no_grad():
            for _ in range(self.n_mc_samples):
                preds_scaled_mc.append(self.model(X_tensor).cpu().numpy().ravel())

        preds_scaled_mc = np.stack(preds_scaled_mc, axis=0)
        mean_scaled = preds_scaled_mc.mean(axis=0)
        std_scaled = preds_scaled_mc.std(axis=0)

        scale_y = float(self.y_scaler.scale_[0])
        center_y = float(self.y_scaler.center_[0])

        mean = mean_scaled * scale_y + center_y
        if not return_std:
            return mean

        std = std_scaled * abs(scale_y)
        return mean, std


# ---------------------------------------------------------
# 3b. Ensemble wrapper (robustness vs “emergent” quirks)
# ---------------------------------------------------------
@dataclass
class EnsembleSurrogate:
    members: List[MCDropoutSurrogate]

    def fit(self, X: np.ndarray, y: np.ndarray, lower: np.ndarray, upper: np.ndarray):
        for m in self.members:
            m.fit(X, y, lower, upper)

    def predict(self, X: np.ndarray, lower: np.ndarray, upper: np.ndarray, return_std: bool = False):
        mus = []
        sigs = []
        for m in self.members:
            mu, sig = m.predict(X, lower, upper, return_std=True)
            mus.append(mu)
            sigs.append(sig)

        mus = np.stack(mus, axis=0)   # [M, N]
        sigs = np.stack(sigs, axis=0) # [M, N]

        # Total uncertainty (law of total variance):
        # Var_total = E[Var] + Var(E)
        mu_ens = mus.mean(axis=0)
        var_within = (sigs ** 2).mean(axis=0)
        var_between = mus.var(axis=0)
        sig_ens = np.sqrt(np.maximum(var_within + var_between, 1e-12))

        if not return_std:
            return mu_ens
        return mu_ens, sig_ens


# ---------------------------------------------------------
# 4. Acquisition helpers
# ---------------------------------------------------------
def acquisition_pi_ei(mu: np.ndarray, sigma: np.ndarray, y_best: float, xi: float = 0.0):
    sigma = np.maximum(sigma, 1e-9)
    gamma = (mu - y_best - xi) / sigma
    pi = norm.cdf(gamma)
    ei = (mu - y_best - xi) * pi + sigma * norm.pdf(gamma)
    return pi, np.maximum(ei, 0.0)


# ---------------------------------------------------------
# 5. Week 9 proposer (global+local + risk-aware hybrid score)
# ---------------------------------------------------------
def propose_next_point_week9(
    surrogate: EnsembleSurrogate,
    X_obs: np.ndarray,
    y_obs: np.ndarray,
    bounds: Tuple[np.ndarray, np.ndarray],
    random_state: int = 123,
    # Exploration knobs (scaled down because n=18 -> diminishing returns from brute force)
    n_sobol: int = 90_000,
    n_local: int = 70_000,
    local_sigma_big: float = 0.10,
    local_sigma_small: float = 0.035,
    # Constraints
    min_dist: float = 0.010,
    repulse_len: float = 0.050,
    repulse_w: float = 0.30,
    # Acquisition mix
    xi_base: float = 0.0015,
    alpha_ucb: float = 0.35,   # 0=only EI, 1=only UCB-style
    # Safety / emergence guard
    bound_margin: float = 0.002,  # discourage hugging bounds
    enforce_rounded_nonduplicate: bool = True
) -> Dict:
    rng = np.random.RandomState(random_state)
    lower, upper = np.asarray(bounds[0], float), np.asarray(bounds[1], float)

    # Clip observed for stable distance + best selection
    Xc = clip_to_bounds(X_obs, lower, upper)

    best_idx = int(np.argmax(y_obs))
    y_best = float(y_obs[best_idx])
    x_best = Xc[best_idx]

    d = Xc.shape[1]
    n = len(y_obs)

    # --- Scaling law intuition (encoded): exploration decays with n ---
    # With 18 points, big-compute scaling tends to show diminishing returns;
    # we reduce xi as n grows, but never to zero.
    xi = float(xi_base + 0.01 / np.sqrt(max(n, 1)))

    # UCB beta: mild, log growth; avoids “sudden” overly-exploratory picks
    beta = float(0.6 * np.sqrt(np.log(n + 2.0)))

    # --- Candidate generation: Sobol global + local trust region around best ---
    # Sobol for coverage (robust to surrogate misspec), local Gaussians for exploitation.
    sob = torch.quasirandom.SobolEngine(dimension=d, scramble=True, seed=int(random_state))
    X_sob = sob.draw(n_sobol).cpu().numpy()
    X_sob = lower + (upper - lower) * X_sob

    X1 = x_best + rng.normal(0.0, local_sigma_big, size=(n_local // 2, d))
    X2 = x_best + rng.normal(0.0, local_sigma_small, size=(n_local - n_local // 2, d))
    X_loc = np.vstack([X1, X2])

    Xcand = clip_to_bounds(np.vstack([X_sob, X_loc]), lower, upper)

    mu, sig = surrogate.predict(Xcand, lower, upper, return_std=True)
    pi, ei = acquisition_pi_ei(mu, sig, y_best=y_best, xi=xi)

    # --- Hard distance constraint + soft repulsion ---
    dmat = np.linalg.norm(Xcand[:, None, :] - Xc[None, :, :], axis=2)
    dmin = dmat.min(axis=1)
    ok = dmin >= min_dist

    penalty = np.exp(-(dmin ** 2) / (2.0 * repulse_len ** 2))

    # --- Soft bound margin penalty (avoid “edge hallucinations” / instability) ---
    # Penalize points too close to bounds (a common emergent failure mode for surrogates).
    dist_to_lower = (Xcand - lower)
    dist_to_upper = (upper - Xcand)
    bound_close = np.minimum(dist_to_lower, dist_to_upper).min(axis=1)  # min margin to any bound
    bound_pen = np.exp(- (bound_close / max(bound_margin, 1e-9))**2 )

    # --- Hybrid score: EI + alpha*(UCB_improvement) - repulsions ---
    # UCB improvement relative to best: (mu + beta*sig - y_best)+
    ucb_impr = np.maximum(mu + beta * sig - y_best, 0.0)

    # Normalize EI for stable mixing
    ei_max = float(np.max(ei[np.isfinite(ei)]) if np.isfinite(ei).any() else 1.0)
    ei_norm = ei / max(ei_max, 1e-12)

    score = (1.0 - alpha_ucb) * ei_norm + alpha_ucb * ucb_impr
    score = score - repulse_w * penalty - 0.08 * bound_pen
    score = np.where(ok, score, -np.inf)

    # Relax once if needed
    if not np.isfinite(score).any():
        ok2 = dmin >= (0.5 * min_dist)
        score = np.where(ok2, (1.0 - alpha_ucb) * ei_norm + alpha_ucb * ucb_impr - repulse_w * penalty - 0.08 * bound_pen, -np.inf)

    next_idx = int(np.argmax(score))
    next_x = Xcand[next_idx].copy()
    next_x_6 = round6(next_x)

    # Ensure rounded point is not a duplicate
    pick_mode = "Week9: Sobol+Local + EI/UCB hybrid"
    if enforce_rounded_nonduplicate:
        X_obs_6 = round6(clip_to_bounds(X_obs, lower, upper))
        seen = set(map(tuple, X_obs_6))
        if tuple(next_x_6) in seen:
            order = np.argsort(score)[::-1]
            found = False
            for j in order[:8000]:
                cand6 = round6(Xcand[j])
                if tuple(cand6) not in seen:
                    next_idx = int(j)
                    next_x = Xcand[next_idx].copy()
                    next_x_6 = cand6
                    pick_mode += " + nondup-fallback"
                    found = True
                    break
            if not found:
                pick_mode += " + DUPLICATE-RO"

    next_mu = float(mu[next_idx])
    next_sig = float(sig[next_idx])

    # nearest neighbors for reasoning
    dists = np.linalg.norm(Xc - next_x, axis=1)
    nn = np.argsort(dists)[:3]

    reasoning = [
        f"• Bounds: lower={lower.round(3)}, upper={upper.round(3)}",
        f"• Current best (observed): y_best={y_best:.6f} at x_best={round6(x_best)} (X clipped).",
        "• Week 9 strategy (scaling/emergence-aware):",
        "   - small ensemble MC-dropout surrogate to reduce variance / brittle behaviour",
        "   - candidate pool: global Sobol coverage + local trust region near best",
        f"   - diminishing-returns exploration: xi={xi:.6f} (decays with n={n})",
        f"   - risk-aware improvement: beta={beta:.4f} in (mu + beta*sigma)",
        f"   - hard min distance={min_dist}, soft repulsion w={repulse_w}, len={repulse_len}",
        f"   - bound-margin penalty to avoid edge-instability (margin={bound_margin})",
        f"   - selection mode: {pick_mode}",
        f"• Pick (raw): x_next={next_x}, μ={next_mu:.6f}, σ={next_sig:.6f}, PI={float(pi[next_idx]):.4f}, EI={float(ei[next_idx]):.6f}",
        f"• Pick (6dp): x_next_6dp={next_x_6}",
        "• Nearest tested points:"
    ]
    for i, idx in enumerate(nn, 1):
        reasoning.append(f"   #{i}: x={round6(X_obs[idx])}, y={float(y_obs[idx]):.6f}, dist={float(dists[idx]):.6f}")

    return dict(
        next_x=next_x_6,
        next_x_raw=next_x,
        pred_mean=next_mu,
        pred_std=next_sig,
        pi=float(pi[next_idx]),
        ei=float(ei[next_idx]),
        y_best=y_best,
        x_best=round6(X_obs[best_idx]),
        reasoning="\n".join(reasoning),
    )


# ---------------------------------------------------------
# 6. Hyperparameter tuning (trimmed for Week 9 cost tradeoff)
# ---------------------------------------------------------
def cv_mse_score(
    config: Dict,
    X: np.ndarray,
    y: np.ndarray,
    bounds: Tuple[np.ndarray, np.ndarray],
    k: int = 5,
    seed: int = 0
) -> float:
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    mses = []
    lower, upper = bounds

    for tr_idx, va_idx in kf.split(X):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        surr = MCDropoutSurrogate(
            hidden_layer_sizes=config["hidden"],
            dropout=config["dropout"],
            n_epochs=config["epochs"],
            lr=config["lr"],
            weight_decay=config["weight_decay"],
            random_state=seed,
            n_mc_samples=config["n_mc_samples"],
            robust_y=True
        )

        surr.fit(Xtr, ytr, lower, upper)
        preds = surr.predict(Xva, lower, upper)
        mses.append(np.mean((preds - yva) ** 2))

    return float(np.mean(mses))


def tune_hyperparameters(
    X: np.ndarray,
    y: np.ndarray,
    bounds: Tuple[np.ndarray, np.ndarray],
    random_state: int = 8
) -> Dict:
    rng = np.random.RandomState(random_state)

    # Week 9: fewer configs (cost) + keep dropout for epistemic uncertainty
    hidden_options = [(64, 32), (64, 64), (128, 64)]
    dropout_options = [0.10, 0.15, 0.20]
    lr_options = [7e-4, 1e-3, 2e-3]
    wd_options = [0.0, 1e-6, 1e-5]

    n_initial = 10
    configs = []
    for _ in range(n_initial):
        cfg = dict(
            hidden=hidden_options[rng.randint(len(hidden_options))],
            dropout=float(dropout_options[rng.randint(len(dropout_options))]),
            lr=float(lr_options[rng.randint(len(lr_options))]),
            weight_decay=float(wd_options[rng.randint(len(wd_options))]),
            n_mc_samples=int([96, 128][rng.randint(2)]),
        )
        configs.append(cfg)

    stage_epochs = [450, 1050]
    keep_fracs = [0.5, 0.4]

    best_overall = None

    for stage, (epochs, keep_frac) in enumerate(zip(stage_epochs, keep_fracs), start=1):
        scored = []
        for cfg in configs:
            cfg_stage = dict(cfg)
            cfg_stage["epochs"] = epochs
            mse = cv_mse_score(cfg_stage, X, y, bounds=bounds, k=5, seed=0)
            scored.append((mse, cfg_stage))

        scored.sort(key=lambda t: t[0])
        if best_overall is None or scored[0][0] < best_overall[0]:
            best_overall = scored[0]

        k_keep = max(4, int(len(scored) * keep_frac))
        configs = [cfg for _, cfg in scored[:k_keep]]

        print(f"\n--- WEEK 9 TUNING STAGE {stage} ---")
        print(f"epochs={epochs}, kept={k_keep}/{len(scored)}")
        print(f"best CV-MSE so far: {best_overall[0]:.6f}")
        print(f"best config so far: {best_overall[1]}")

    return dict(best_cv_mse=best_overall[0], best_config=best_overall[1])


# ---------------------------------------------------------
# 7. Main: tune -> fit ensemble -> propose next (6 decimals)
# ---------------------------------------------------------
def main():
    np.random.seed(0)
    torch.manual_seed(0)

    bounds = (np.zeros(3), np.ones(3))

    # Tune hyperparameters (cost-trimmed)
    tuning = tune_hyperparameters(X_raw, y_raw, bounds=bounds, random_state=8)
    best_cfg = tuning["best_config"]

    # Fit a small ensemble (robustness vs emergent, uneven behaviour)
    seeds = [0, 11, 29]  # fixed for determinism
    members = []
    for s in seeds:
        members.append(
            MCDropoutSurrogate(
                hidden_layer_sizes=best_cfg["hidden"],
                dropout=best_cfg["dropout"],
                n_epochs=best_cfg["epochs"],
                lr=best_cfg["lr"],
                weight_decay=best_cfg["weight_decay"],
                random_state=s,
                n_mc_samples=best_cfg["n_mc_samples"],
                robust_y=True
            )
        )

    surrogate = EnsembleSurrogate(members=members)
    surrogate.fit(X_raw, y_raw, bounds[0], bounds[1])

    # Current best observed
    best_idx = int(np.argmax(y_raw))
    current_best_x = round6(X_raw[best_idx])
    current_best_y = float(y_raw[best_idx])

    # Propose next query within [0,1]^3
    suggestion = propose_next_point_week9(
        surrogate=surrogate,
        X_obs=X_raw,
        y_obs=y_raw,
        bounds=bounds,
        random_state=123,
        n_sobol=90_000,
        n_local=70_000,
        local_sigma_big=0.10,
        local_sigma_small=0.035,
        min_dist=0.010,
        repulse_len=0.050,
        repulse_w=0.30,
        xi_base=0.0015,
        alpha_ucb=0.35,
        bound_margin=0.002,
        enforce_rounded_nonduplicate=True
    )

    print("\n================================================")
    print("WEEK 9 FUNCTION 3 — NEXT POINT (6 DECIMALS)")
    print("================================================")
    print("Surrogate: ensemble(mc_dropout) + robust y scaling")
    print(f"Best CV-MSE (lower is better): {tuning['best_cv_mse']:.6f}")
    print("Best tuned hyperparameters:")
    print(f"  hidden: {best_cfg['hidden']}")
    print(f"  dropout: {best_cfg['dropout']}")
    print(f"  lr: {best_cfg['lr']}")
    print(f"  weight_decay: {best_cfg['weight_decay']}")
    print(f"  n_mc_samples: {best_cfg['n_mc_samples']}")
    print(f"  epochs: {best_cfg['epochs']}")
    print(f"Ensemble seeds: {seeds}")

    print("\n================================================")
    print("CURRENT BEST OBSERVED")
    print("================================================")
    print(f"x_best = {as_fixed6_list(current_best_x)}, y_best = {current_best_y:.6f}")

    print("\n================================================")
    print("RECOMMENDED NEXT POINT (rounded to 6 decimals)")
    print("================================================")
    x_next = suggestion["next_x"]
    print(f"x_next     = [{x_next[0]:.6f}, {x_next[1]:.6f}, {x_next[2]:.6f}]")
    print(f"μ(x_next)  = {suggestion['pred_mean']:.6f}")
    print(f"σ(x_next)  = {suggestion['pred_std']:.6f}")
    print(f"PI         = {suggestion['pi']:.4f}")
    print(f"EI         = {suggestion['ei']:.6f}")

    print("\n================================================")
    print("REASONING")
    print("================================================")
    print(suggestion["reasoning"])


if __name__ == "__main__":
    main()



--- WEEK 9 TUNING STAGE 1 ---
epochs=450, kept=5/10
best CV-MSE so far: 0.032987
best config so far: {'hidden': (64, 32), 'dropout': 0.2, 'lr': 0.0007, 'weight_decay': 1e-06, 'n_mc_samples': 128, 'epochs': 450}

--- WEEK 9 TUNING STAGE 2 ---
epochs=1050, kept=4/5
best CV-MSE so far: 0.032987
best config so far: {'hidden': (64, 32), 'dropout': 0.2, 'lr': 0.0007, 'weight_decay': 1e-06, 'n_mc_samples': 128, 'epochs': 450}

WEEK 9 FUNCTION 3 — NEXT POINT (6 DECIMALS)
Surrogate: ensemble(mc_dropout) + robust y scaling
Best CV-MSE (lower is better): 0.032987
Best tuned hyperparameters:
  hidden: (64, 32)
  dropout: 0.2
  lr: 0.0007
  weight_decay: 1e-06
  n_mc_samples: 128
  epochs: 450
Ensemble seeds: [0, 11, 29]

CURRENT BEST OBSERVED
x_best = [0.403756, 0.381706, 0.489738], y_best = -0.009136

RECOMMENDED NEXT POINT (rounded to 6 decimals)
x_next     = [0.846635, 0.969038, 0.008713]
μ(x_next)  = -0.056213
σ(x_next)  = 0.035159
PI         = 0.0748
EI         = 0.001177

REASONING
• Bounds